In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from skimage.feature import hog
from skimage.color import rgb2gray


print('Numpy:', np.__version__)
print('Pandas:', pd.__version__)

Numpy: 2.4.2
Pandas: 3.0.1


In [2]:
DATA_DIR = "../Dataset/bloodcells_dataset"

classes = sorted(os.listdir(DATA_DIR))
print('Classes trouvées :', classes)
print('Nombre de classes :', len(classes))

Classes trouvées : ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']
Nombre de classes : 8


In [3]:
def center_crop(img, crop_size=200):
    w, h = img.size
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size
    return img.crop((left, top, right, bottom))

def apply_circular_mask(img_array):
    h, w = img_array.shape[:2]
    cy, cx = h // 2, w // 2
    radius = min(h, w) // 2
    Y, X = np.ogrid[:h, :w]
    mask = (X - cx)**2 + (Y - cy)**2 <= radius**2
    result = img_array.copy()
    result[~mask] = 0
    return result


X = []
Y = []

for label, cls in enumerate(classes):
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_dir):
        img = Image.open(os.path.join(cls_dir, fname))
        img = center_crop(img, crop_size=200)
        img = np.array(img.resize((64, 64))) / 255.0
        img = apply_circular_mask(img)
        X.append(img)
        Y.append(label)

X = np.array(X)
Y = np.array(Y)

print(f"X shape: {X.shape}, dtype: {X.dtype}")
print(f"Y shape: {Y.shape}")


X shape: (17092, 64, 64, 3), dtype: float64
Y shape: (17092,)


In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

X_train_flat = X_train.reshape(X_train.shape[0], -1)  # (13673, 12288)
X_test_flat = X_test.reshape(X_test.shape[0], -1)     # (3419, 12288)

print(f"Train flat: {X_train_flat.shape}")
print(f"Test flat: {X_test_flat.shape}")


Train: (13673, 64, 64, 3), Test: (3419, 64, 64, 3)
Train flat: (13673, 12288)
Test flat: (3419, 12288)


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y)

# HOG features
def extract_hog_features(img_rgb):
    gray = rgb2gray(img_rgb)
    return hog(gray, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm='L2-Hys')

X_train_hog = np.array([extract_hog_features(img) for img in X_train])
X_test_hog = np.array([extract_hog_features(img) for img in X_test])

# addition des deux featuress
X_train_flat = np.hstack([X_train.reshape(X_train.shape[0], -1), X_train_hog])
X_test_flat = np.hstack([X_test.reshape(X_test.shape[0], -1), X_test_hog])

print(f"Train combined: {X_train_flat.shape}")
print(f"Test combined: {X_test_flat.shape}")


Train combined: (13673, 14052)
Test combined: (3419, 14052)


In [10]:
randforest3 = RandomForestClassifier(
    n_estimators=400,
    max_depth=30,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
randforest3.fit(X_train_flat, Y_train)

Y_pred3 = randforest3.predict(X_test_flat)
print(f"Accuracy: {accuracy_score(Y_test, Y_pred3):.4f}")
print(classification_report(Y_test, Y_pred3, target_names=classes))


Accuracy: 0.8742
              precision    recall  f1-score   support

    basophil       0.87      0.61      0.72       244
  eosinophil       0.97      0.95      0.96       623
erythroblast       0.94      0.84      0.89       310
          ig       0.68      0.83      0.75       579
  lymphocyte       0.83      0.85      0.84       243
    monocyte       0.87      0.66      0.75       284
  neutrophil       0.90      0.97      0.93       666
    platelet       1.00      1.00      1.00       470

    accuracy                           0.87      3419
   macro avg       0.88      0.84      0.85      3419
weighted avg       0.88      0.87      0.87      3419



Effectivemtn on cosnate un meilleurs résultat de base avec  qui peut surement etre optimiser :


In [16]:
randforest3 = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
randforest3.fit(X_train_flat, Y_train)

Y_pred3 = randforest3.predict(X_test_flat)
print(f"Accuracy: {accuracy_score(Y_test, Y_pred3):.4f}")
print(classification_report(Y_test, Y_pred3, target_names=classes))


Accuracy: 0.8754
              precision    recall  f1-score   support

    basophil       0.88      0.64      0.74       244
  eosinophil       0.97      0.95      0.96       623
erythroblast       0.94      0.84      0.89       310
          ig       0.68      0.83      0.75       579
  lymphocyte       0.84      0.86      0.85       243
    monocyte       0.86      0.66      0.75       284
  neutrophil       0.90      0.97      0.93       666
    platelet       1.00      1.00      1.00       470

    accuracy                           0.88      3419
   macro avg       0.88      0.84      0.86      3419
weighted avg       0.88      0.88      0.87      3419



Test Avec Hog + Color Hist (sans le flat global )

La difference entre le flat layer qu'on utilisais avant et les hsitogramme hsv c'tes que on ne prend plus en compte la position de la couelrus mais surtout la distribution de la couleur global : ici on se fiche de perdre la localisation et les forme du flat layer car elle sont deja prise en compte dans le hog

In [ ]:
from skimage.color import rgb2hsv

def extract_hog_features(img_rgb):
    gray = rgb2gray(img_rgb)
    return hog(gray, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm='L2-Hys')

def extract_color_hist(img_rgb, bins=32):
    hsv = rgb2hsv(img_rgb)
    hists = []
    for c in range(3):
        h, _ = np.histogram(hsv[:,:,c], bins=bins, range=(0, 1))
        h = h / (h.sum() + 1e-8)
        hists.append(h)
    return np.concatenate(hists)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y)

X_train_hog = np.array([extract_hog_features(img) for img in X_train])
X_test_hog = np.array([extract_hog_features(img) for img in X_test])

X_train_color = np.array([extract_color_hist(img) for img in X_train])
X_test_color = np.array([extract_color_hist(img) for img in X_test])

# juste hog + hst couleur
X_train_combined = np.hstack([X_train_hog, X_train_color])
X_test_combined = np.hstack([X_test_hog, X_test_color])

print(f"HOG: {X_train_hog.shape[1]}, Color: {X_train_color.shape[1]}, Total: {X_train_combined.shape[1]}")


HOG: 1764, Color: 96, Total: 1860


In [19]:
randforest3 = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
randforest3.fit(X_train_combined, Y_train)

Y_pred3 = randforest3.predict(X_test_combined)
print(f"Accuracy: {accuracy_score(Y_test, Y_pred3):.4f}")
print(classification_report(Y_test, Y_pred3, target_names=classes))


Accuracy: 0.9213
              precision    recall  f1-score   support

    basophil       0.96      0.88      0.92       244
  eosinophil       0.99      0.99      0.99       623
erythroblast       0.97      0.82      0.89       310
          ig       0.78      0.86      0.82       579
  lymphocyte       0.91      0.95      0.93       243
    monocyte       0.93      0.78      0.85       284
  neutrophil       0.90      0.96      0.93       666
    platelet       1.00      1.00      1.00       470

    accuracy                           0.92      3419
   macro avg       0.93      0.91      0.92      3419
weighted avg       0.93      0.92      0.92      3419



fin du projet lol